In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../dataset_wirelength/sky130/ac97_top/CLK_7.8_IO_0.00_CU_20_AR_0.5_HW_4_HS_4_HP_32_VW_4_VS_4_VP_32/nets.csv')

In [4]:
df.columns

Index(['net', 'driving_cell', 'driving_cell_area', 'driving_cell_strength',
       'driving_inst', 'fanin', 'fanout_area', 'is_buffer', 'is_inverter',
       'is_port', 'pin_capacitance', 'term_bbox_ar', 'term_bbox_area',
       'term_bbox_dx', 'term_bbox_dy', 'term_bbox_hpwl', 'term_bbox_xmax',
       'term_bbox_xmin', 'term_bbox_ymax', 'term_bbox_ymin', 'terms',
       'wire_capacitance_placement', 'wirelength'],
      dtype='str')

In [4]:
# 1. Считаем матрицу корреляции для всего DataFrame
matrix = df.corr(numeric_only=True)

# 2. Выделяем столбец нашей метки (например, 'Цена') 
# и сортируем значения от самой сильной положительной связи к отрицательной
target_corr = matrix['wirelength'].sort_values(ascending=False)

In [6]:
matrix.style.background_gradient(cmap='coolwarm')

,driving_cell_area,fanin,fanout_area,is_buffer,is_inverter,is_port,pin_capacitance,term_bbox_ar,term_bbox_area,term_bbox_dx,term_bbox_dy,term_bbox_hpwl,term_bbox_xmax,term_bbox_xmin,term_bbox_ymax,term_bbox_ymin,terms,wire_capacitance_placement,wirelength
driving_cell_area,1.000000,0.340020,0.018691,-0.422153,0.112149,-0.203990,-0.180455,-0.039589,-0.097171,-0.111297,-0.093625,-0.117135,-0.006720,0.014065,-0.133550,-0.108123,-0.051262,-0.117211,-0.128392
fanin,0.340020,1.000000,-0.349292,-0.525583,-0.302569,-0.263945,-0.355345,0.006845,-0.187236,-0.249419,-0.229105,-0.273865,0.040947,0.087302,-0.131332,-0.068975,-0.373821,-0.297068,-0.301091
fanout_area,0.018691,-0.349292,1.000000,0.353268,0.361022,-0.080144,0.500267,-0.037186,0.450846,0.518181,0.488952,0.576640,0.025414,-0.071337,0.016912,-0.116349,0.863792,0.667957,0.665242
is_buffer,-0.422153,-0.525583,0.353268,1.000000,-0.131248,-0.067771,0.276351,0.051430,0.320295,0.419396,0.358652,0.444804,0.070058,-0.008448,0.130941,0.033274,0.449406,0.468664,0.471984
is_inverter,0.112149,-0.302569,0.361022,-0.131248,1.000000,-0.039014,0.496832,-0.019868,0.002387,0.063469,0.047405,0.063246,-0.019932,-0.031690,-0.015251,-0.028183,0.277148,0.080870,0.064421
is_port,-0.203990,-0.263945,-0.080144,-0.067771,-0.039014,1.000000,-0.059563,-0.011373,-0.020205,-0.024128,-0.024167,-0.027659,-0.142189,-0.137117,0.055061,0.061686,-0.069233,-0.033303,-0.016010
pin_capacitance,-0.180455,-0.355345,0.500267,0.276351,0.496832,-0.059563,1.000000,-0.028914,0.240015,0.330811,0.309580,0.366594,0.012592,-0.049160,0.032824,-0.051535,0.555302,0.430942,0.424968
term_bbox_ar,-0.039589,0.006845,-0.037186,0.051430,-0.019868,-0.011373,-0.028914,1.000000,-0.017796,0.175842,-0.046968,0.070048,0.005383,-0.027435,0.018370,0.031185,-0.035290,0.046857,0.044185
term_bbox_area,-0.097171,-0.187236,0.450846,0.320295,0.002387,-0.020205,0.240015,-0.017796,1.000000,0.698059,0.753975,0.833035,0.095690,-0.034894,0.098713,-0.106729,0.478650,0.830526,0.826006
term_bbox_dx,-0.111297,-0.249419,0.518181,0.419396,0.063469,-0.024128,0.330811,0.175842,0.698059,1.000000,0.522251,0.864187,0.071999,-0.114806,0.112014,-0.030258,0.602911,0.848060,0.846399


In [7]:
target_corr

wirelength                    1.000000
wire_capacitance_placement    0.997628
term_bbox_hpwl                0.970281
term_bbox_dy                  0.846463
term_bbox_dx                  0.846399
term_bbox_area                0.826006
terms                         0.758146
fanout_area                   0.665242
is_buffer                     0.471984
pin_capacitance               0.424968
term_bbox_ymax                0.122818
is_inverter                   0.064421
term_bbox_xmax                0.055312
term_bbox_ar                  0.044185
is_port                      -0.016010
term_bbox_xmin               -0.107354
term_bbox_ymin               -0.113229
driving_cell_area            -0.128392
fanin                        -0.301091
Name: wirelength, dtype: float64

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

FEATURES = [ 
    'driving_cell_area', 
    'driving_cell_strength',
    'fanin',
    'fanout_area',
    'is_buffer',
    'is_inverter',
    'is_port', 
    'pin_capacitance',
    'term_bbox_ar',
    'term_bbox_area',
    'term_bbox_hpwl',
    'terms',
    'wire_capacitance_placement'
]

TARGET = "wirelength"
DATASET = Path("/home/nvgel/phd/dataset_wirelength/")

# First experiment: one PDK.
files = sorted((DATASET / "sky130").rglob("net_wirelength.parquet"))
if not files:
    raise FileNotFoundError("No Parquet files found")

df = pd.concat(
    [pd.read_parquet(p) for p in files],
    ignore_index=True,
)

required = FEATURES + [TARGET, "design"]
df = df.dropna(subset=required)
df = df[np.isfinite(df[FEATURES + [TARGET]]).all(axis=1)]

# One fold for an initial smoke test.
cv = GroupKFold(n_splits=5)
train_idx, test_idx = next(
    cv.split(df, groups=df["design"])
)

train = df.iloc[train_idx]
test = df.iloc[test_idx]

# Sample training nets only; never sample the test blocks.
n_train = min(10000, len(train))
train = train.sample(n=n_train, random_state=42)

X_train = train[FEATURES].to_numpy(dtype=np.float32)
y_train = train[TARGET].to_numpy(dtype=np.float32)
X_test = test[FEATURES].to_numpy(dtype=np.float32)
y_test = test[TARGET].to_numpy(dtype=np.float32)

models = {
    "XGBoost": XGBRegressor(
        n_estimators=700,
        max_depth=8,
        learning_rate=0.05,
        tree_method="hist",
        n_jobs=12,
        random_state=42,
    ),
    "TabPFN": TabPFNRegressor(device="auto"),
}

for name, model in models.items():
    model.fit(X_train, y_train)

    # Batch prediction for large test datasets.
    predictions = []
    for start in range(0, len(X_test), 2000):
        batch = X_test[start:start + 2000]
        predictions.append(model.predict(batch))

    y_pred = np.concatenate(predictions)
    y_pred = np.maximum(y_pred, 0.0)

    print(
        f"{name}: "
        f"MAE={mean_absolute_error(y_test, y_pred):.4f}, "
        f"R2={r2_score(y_test, y_pred):.4f}"
    )

TypeError: ufunc 'isfinite' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [ ]:
df.shape

In [2]:
print(df[FEATURES].dtypes)

for col in FEATURES:
    if df[col].dtype == "object":
        print(f"\nColumn: {col}")
        print(df[col].dropna().unique()[:20])

driving_cell_area             float64
driving_cell_strength             str
fanin                           int64
fanout_area                   float64
is_buffer                       int64
is_inverter                     int64
is_port                         int64
pin_capacitance               float64
term_bbox_ar                  float64
term_bbox_area                float64
term_bbox_hpwl                float64
terms                           int64
wire_capacitance_placement    float64
dtype: object


In [3]:
import numpy as np
import pandas as pd

FEATURES = [
    "driving_cell_area",
    "driving_cell_strength",
    "fanin",
    "fanout_area",
    "is_buffer",
    "is_inverter",
    "is_port",
    "pin_capacitance",
    "term_bbox_ar",
    "term_bbox_area",
    "term_bbox_hpwl",
    "terms",
    "wire_capacitance_placement",
]

TARGET = "wirelength"

# Work on a copy
df = df.copy()

# Convert boolean features to 0/1
BOOL_FEATURES = ["is_buffer", "is_inverter", "is_port"]

for col in BOOL_FEATURES:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({
            "true": 1,
            "false": 0,
            "1": 1,
            "0": 0,
        })
    )

# Convert all features to numeric
for col in FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

# Replace infinities with NaN
df[FEATURES + [TARGET]] = (
    df[FEATURES + [TARGET]]
    .replace([np.inf, -np.inf], np.nan)
)

# Diagnose invalid values
print("Missing/invalid values:")
print(df[FEATURES + [TARGET]].isna().sum())

# For the initial experiment, remove invalid rows
df = df.dropna(subset=FEATURES + [TARGET])

print(f"Remaining nets: {len(df)}")

Missing/invalid values:
driving_cell_area                  0
driving_cell_strength         122225
fanin                              0
fanout_area                        0
is_buffer                          0
is_inverter                        0
is_port                            0
pin_capacitance                    0
term_bbox_ar                       0
term_bbox_area                     0
term_bbox_hpwl                     0
terms                              0
wire_capacitance_placement         0
wirelength                         0
dtype: int64
Remaining nets: 10053501
